In [9]:
from pathlib import Path
import subprocess
import sys

In [43]:
sys.executable

'd:\\study-on-agent\\.venv\\Scripts\\python.exe'

In [ ]:
def run_script(directory:Path, script:str, args: list[str])->str:
    script_path = directory / script
    if not script_path.is_file():
        raise FileNotFoundError(f"no such script in skill: {script_path}")
    result = subprocess.run(
        [sys.executable, str(script_path), *(args or [])],
        capture_output=True,
        text=True,

    )
    return result

In [38]:
ROOT = Path.cwd().parent
TOOL_DIR = ROOT / "dev_tools"
SKILL_DIR = ROOT / "dev_skills"

In [44]:
def to_argv(args: dict) -> list[str]:
    argv = []
    for key, value in args.items():
        argv += [f"--{key}", str(value)]
    return argv

# dispatcher side:
args = to_argv({"file": f"{TOOL_DIR / "read_file.py"}"})
result = run_script(TOOL_DIR, "read_file.py", args)
# -> subprocess runs: python read_file.py --file notes.txt
# -> stdout captured -> that's your content, back in tool_result
print(result.stdout)


import argparse
import sys
from pathlib import Path

DESCRIPTION = "Read a file's contents. Use when the user needs to see what's in a specific file."
ARGS = '{"file": "str"}'  # a plain string here on purpose -- see below

def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('-f', '--file', required=True, help='File to read')
    return parser.parse_args()

def main():
    args = get_args()
    path = Path(args.file)
    if not path.is_file():
        print(f"no such file: {path}", file=sys.stderr)
        sys.exit(1)  # nonzero exit -- run_script folds this into tool_result as a refusal/error, same as delete_file's guardrail would
    print(path.read_text(encoding="utf-8"))  # <-- this is the "return value"

if __name__=='__main__':
    main()




In [45]:
args

['--file', 'd:\\study-on-agent\\dev_tools\\read_file.py']

In [40]:
import ast
import json
from pathlib import Path

def read_tool(path: Path) -> dict:
    tree = ast.parse(path.read_text(encoding="utf-8"))

    meta = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and target.id in ("DESCRIPTION", "ARGS"):
                meta[target.id] = ast.literal_eval(node.value)

    return {
        "name": path.stem,
        "description": meta.get("DESCRIPTION", ""),
        "args": json.loads(meta["ARGS"]) if meta.get("ARGS") else {},
        "path": path,
    }

def read_skill(path: Path) -> dict:
    text = path.read_text(encoding="utf-8")
    _, frontmatter, _body = text.split("---", 2)

    meta = {}
    for line in frontmatter.strip().splitlines():
        key, _, value = line.partition(":")
        meta[key.strip()] = value.strip()

    return {
        "name": meta["name"],
        "description": meta["description"],
        "args": json.loads(meta["args"]) if "args" in meta else {},
        "path": path,
    }


In [41]:
read_tool(TOOL_DIR/"read_file.py")

{'name': 'read_file',
 'description': "Read a file's contents. Use when the user needs to see what's in a specific file.",
 'args': {'file': 'str'},
 'path': WindowsPath('d:/study-on-agent/dev_tools/read_file.py')}

In [42]:
read_skill(SKILL_DIR/"farewell"/"SKILL.md")

{'name': 'farewell',
 'description': 'abc',
 'args': {},
 'path': WindowsPath('d:/study-on-agent/dev_skills/farewell/SKILL.md')}

In [57]:
import argparse

def get_args():
    parser = argparse.ArgumentParser(description="test idea")
    parser.add_argument('-f', '--file', required=True, help='File to read', type=str, default="dummy.txt")
    return parser

In [58]:
_ = get_args()

In [60]:
_._actions[1]

_StoreAction(option_strings=['-f', '--file'], dest='file', nargs=None, const=None, default='dummy.txt', type=<class 'str'>, choices=None, required=True, help='File to read', metavar=None)

In [77]:
from dataclasses import dataclass

@dataclass
class Arg:
    field:str
    description:str
    dtype:type[str]|type[int]|type[float]
    required:bool
    default: str|int|float|None=None

@dataclass
class Args:
    args:list[Arg]

In [79]:
arg = _._actions[1]
Arg(
    field=arg.dest,
    description=arg.help,
    dtype=arg.type,
    required=arg.required,
    default=arg.default
)

Arg(field='file', description='File to read', dtype=<class 'str'>, required=True, default='dummy.txt')

In [72]:
_._actions[1].type is str

True